# ⚡ GPU 版：深度學習分割（Cellpose）能不能再往上推？

示範「**GPU 讓 AI 用得起更強的方法**」：把 CPU 的 watershed 偵測，換成 **GPU 上的深度學習細胞分割 Cellpose**，用**同一把官方尺**三方對照：
`baseline`(CPU 閾值) vs `watershed`(CPU) vs `cellpose`(GPU 深度學習)。

> 需 **GPU + Internet On**（裝 cellpose、下載權重）。本 notebook 不提交。
> 這是**煙霧測試版**：先 1 個樣本、前 8 幀，確認 Cellpose 在 GPU 上跑得起來；通了再放大。

In [ ]:
import numpy, torch, subprocess, sys
NV, TV = numpy.__version__, torch.__version__
def pip(*a):
    print('pip', *a); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a])
# 釘住 numpy+torch：cellpose 會想換掉它們，但 Kaggle 的 torch 才有 P100 的 GPU kernel
pip(f'numpy=={NV}', f'torch=={TV}', 'cellpose>=3,<4')
pip('git+https://github.com/royerlab/tracksdata')
pip('--no-deps', 'git+https://github.com/royerlab/kaggle-cell-tracking-competition')
print('CUDA:', torch.cuda.is_available(), '| numpy', NV, '| torch', TV)
import tracksdata as td, polars as pl
from tracking_cellmot.metrics import evaluate, per_sample_metrics, summarise, node_recall
from cellpose import models as cp_models
print('imports OK')

In [ ]:
import os
from collections import defaultdict
import numpy as np
import zarr
from scipy.ndimage import uniform_filter, label, distance_transform_edt
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

TRAIN = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
SCALE = (1.625, 0.40625, 0.40625); SCALE_A = np.array(SCALE)
DOWNSAMPLE, PERCENTILE = 4, 90
MAX_LINK_DISTANCE, DIV_DISTANCE, GAP_DISTANCE, WS_MIN_DISTANCE = 15.0, 8.0, 20.0, 2
MAX_SAMPLES, MAX_T = 1, 1000        # 代表性：1 樣本、全部時間幀
ANISOTROPY = SCALE[0] / SCALE[1]   # z 比 xy 粗 ~4 倍

def scaled_pairwise(A, B):
    d = A[:, None, :] - B[None, :, :]; return np.sqrt(((d * SCALE_A) ** 2).sum(axis=2))
def open_image(p): return zarr.open(p, mode='r')['0']
def read_estimated_n_total(geff_path):
    try:
        a = dict(zarr.open(geff_path, mode='r').attrs)
        def walk(d):
            if isinstance(d, dict):
                for k, v in d.items():
                    if 'estimated_number_of_nodes' in str(k): return v
                    r = walk(v)
                    if r is not None: return r
            return None
        v = walk(a)
        if v is not None: return float(v)
    except Exception as e: print('n_total warn:', e)
    return float('nan')
print('readers ready')

In [ ]:
# Cellpose 模型只建一次（GPU）
CP = cp_models.Cellpose(gpu=True, model_type='nuclei')

def detect_baseline(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    b = sm > np.percentile(sm, PERCENTILE)
    lab, n = label(b)
    return [np.argwhere(lab == i).mean(0) * DOWNSAMPLE for i in range(1, n + 1) if (lab == i).any()]

def detect_watershed(vol):
    ds = vol[::DOWNSAMPLE, ::DOWNSAMPLE, ::DOWNSAMPLE]
    sm = uniform_filter(ds.astype(np.float32), size=3)
    b = sm > np.percentile(sm, PERCENTILE)
    if not b.any(): return []
    dist = distance_transform_edt(b)
    pk = peak_local_max(dist, min_distance=WS_MIN_DISTANCE, labels=b)
    if len(pk) == 0: return []
    mk = np.zeros(dist.shape, np.int32); mk[tuple(pk.T)] = np.arange(1, len(pk) + 1)
    lab = watershed(-dist, mk, mask=b)
    return [np.argwhere(lab == i).mean(0) * DOWNSAMPLE for i in range(1, int(lab.max()) + 1) if (lab == i).any()]

def detect_cellpose(vol):
    # 降採樣 2 倍給 Cellpose 提速；質心再乘回去
    ds = vol[::2, ::2, ::2].astype(np.float32)
    masks, flows, styles, diams = CP.eval(ds, do_3D=True, channels=[0, 0],
                                          anisotropy=ANISOTROPY, diameter=8)
    out = []
    for i in range(1, int(masks.max()) + 1):
        c = np.argwhere(masks == i)
        if len(c): out.append(c.mean(0) * 2)
    return out

def run_pipeline(arr, n_t, detect_fn, improved):
    nodes, edges = {}, []
    frame_ids, frame_xyz = [], []
    nid = 1
    for t in range(n_t):
        cents = detect_fn(np.asarray(arr[t]))
        ids, xyz = [], []
        for c in cents:
            nodes[nid] = (t, float(c[0]), float(c[1]), float(c[2])); ids.append(nid); xyz.append(c); nid += 1
        frame_ids.append(ids); frame_xyz.append(np.array(xyz) if xyz else np.empty((0, 3)))
    has_in, out_count = set(), defaultdict(int)
    for t in range(n_t - 1):
        pid, pc = frame_ids[t], frame_xyz[t]; cid, cc = frame_ids[t + 1], frame_xyz[t + 1]
        if len(pid) == 0 or len(cid) == 0: continue
        D = scaled_pairwise(pc, cc); rr, c2 = linear_sum_assignment(D)
        mp, mc = set(), set()
        for ri, ci in zip(rr, c2):
            if D[ri, ci] <= MAX_LINK_DISTANCE:
                edges.append((pid[ri], cid[ci])); mp.add(ri); mc.add(ci); has_in.add(cid[ci]); out_count[pid[ri]] += 1
        if improved:
            for ci in range(len(cid)):
                if ci in mc: continue
                dd = np.sqrt((((pc - cc[ci]) * SCALE_A) ** 2).sum(1)); j = int(np.argmin(dd))
                if j in mp and dd[j] <= DIV_DISTANCE and out_count[pid[j]] < 2:
                    edges.append((pid[j], cid[ci])); has_in.add(cid[ci]); out_count[pid[j]] += 1
    if improved:
        has_out = set(out_count.keys())
        for t in range(n_t - 2):
            ends = [(i, n_) for i, n_ in enumerate(frame_ids[t]) if n_ not in has_out]
            starts = [(j, n_) for j, n_ in enumerate(frame_ids[t + 2]) if n_ not in has_in]
            if not ends or not starts: continue
            ec = frame_xyz[t][[i for i, _ in ends]]; sc = frame_xyz[t + 2][[j for j, _ in starts]]
            D = scaled_pairwise(ec, sc); rr, c2 = linear_sum_assignment(D)
            for ri, ci in zip(rr, c2):
                if D[ri, ci] <= GAP_DISTANCE:
                    edges.append((ends[ri][1], starts[ci][1])); has_out.add(ends[ri][1]); has_in.add(starts[ci][1])
    return nodes, edges

def build_graph(nodes, edges):
    items = list(nodes.items()); id2idx = {nid: i for i, (nid, _) in enumerate(items)}
    g = td.graph.InMemoryGraph()
    for key in ['z', 'y', 'x']: g.add_node_attr_key(key, pl.Float64, -999999.0)
    tids = g.bulk_add_nodes([{'t': int(t), 'z': float(z), 'y': float(y), 'x': float(x)} for (_, (t, z, y, x)) in items])
    g.add_edge_attr_key('edge_prob', pl.Float64, 0.0)
    ed = [{'source_id': tids[id2idx[s]], 'target_id': tids[id2idx[d]], 'edge_prob': 1.0} for s, d in edges if s in id2idx and d in id2idx]
    if ed: g.bulk_add_edges(ed)
    return g
print('pipeline + cellpose ready')

In [ ]:
import time
samples = sorted(d[:-5] for d in os.listdir(TRAIN) if d.endswith('.geff'))[:MAX_SAMPLES]
print('樣本：', samples, '| 前', MAX_T, '幀\n')
results = {'baseline': [], 'watershed': [], 'cellpose': []}
for name in samples:
    arr = open_image(os.path.join(TRAIN, name + '.zarr'))
    n_t = min(arr.shape[0], MAX_T)
    gt_res = td.graph.IndexedRXGraph.from_geff(os.path.join(TRAIN, name + '.geff'))
    gt = gt_res[0] if isinstance(gt_res, tuple) else gt_res
    n_total = read_estimated_n_total(os.path.join(TRAIN, name + '.geff'))
    for tag, detect, improved in [('baseline', detect_baseline, False),
                                  ('watershed', detect_watershed, True),
                                  ('cellpose', detect_cellpose, True)]:
        try:
            t0 = time.time()
            nodes, edges = run_pipeline(arr, n_t, detect, improved)
            g = build_graph(nodes, edges)
            er = evaluate(g, gt, scale=SCALE, max_distance=7.0)
            recall = node_recall(g, gt)
            row = per_sample_metrics(er, n_total, recall)
            results[tag].append(row)
            print(f'{name} [{tag:9}] {time.time()-t0:5.1f}s  pred_nodes={len(nodes):6d}  adj_edge_j={row.get("adj_edge_jaccard"):.4f}  recall={row.get("node_recall"):.3f}')
        except Exception as e:
            import traceback; traceback.print_exc(); print(f'{name} [{tag}] FAILED: {e}')

print('\n=== 三方官方分數（前', MAX_T, '幀，1 樣本）===')
for tag in ['baseline', 'watershed', 'cellpose']:
    if results[tag]:
        s = summarise(results[tag]); print(f'{tag:9}: score={s.get("score"):.4f}')